# PHASE 7 — API FastAPI (PoC)
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Créer une API légère qui charge le modèle entraîné et retourne un score de fraude + explication SHAP.

**Jours 39-42 du plan**

---
⚠️ **Rappel de cadrage :** PoC + spécifications + tests, pas un logiciel complet déployé.

In [ ]:
!pip install fastapi uvicorn pydantic shap xgboost scikit-learn pandas numpy joblib

In [ ]:
# === Contenu du fichier API ===
# Ce code sera enregistré dans src/api.py

api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import numpy as np
import joblib
import shap
import uvicorn

# === Chargement du modèle et des artefacts ===
model = joblib.load('models/xgb_model.pkl')
scaler = joblib.load('models/scaler.pkl')
freq_maps = joblib.load('models/freq_maps.pkl')
best_threshold = np.load('models/best_threshold.npy')

try:
    ohe = joblib.load('models/ohe_encoder.pkl')
except:
    ohe = None

# SHAP Explainer
explainer = shap.TreeExplainer(model)

# === Application FastAPI ===
app = FastAPI(
    title="FRAUDX - API Détection de Fraude",
    description="API de détection de fraude bancaire avec explicabilité SHAP",
    version="1.0.0"
)

class Transaction(BaseModel):
    TransactionAmt: float
    card1: float = None
    card2: float = None
    addr1: float = None
    addr2: float = None
    hour: int = None
    dayofweek: int = None
    # Ajouter d'autres champs selon les besoins

class PredictionResponse(BaseModel):
    transaction_id: str
    fraud_score: float
    prediction: str  # "FRAUDE" ou "NORMALE"
    risk_level: str  # "Faible", "Moyen", "Élevé", "Critique"
    top_features: list

def get_risk_level(score: float) -> str:
    if score >= 0.9:
        return "Critique"
    elif score >= 0.7:
        return "Élevé"
    elif score >= 0.4:
        return "Moyen"
    else:
        return "Faible"

@app.get("/")
def read_root():
    return {
        "service": "FRAUDX - Détection de fraude bancaire",
        "version": "1.0.0",
        "status": "opérationnel"
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(transaction: Transaction):
    try:
        # Convertir en DataFrame
        data = pd.DataFrame([transaction.dict()])
        
        # Prétraitement simplifié
        # (Dans une version complète, reproduire tout le pipeline Phase 2)
        
        # Prédiction
        proba = model.predict_proba(data)[0, 1]
        prediction = "FRAUDE" if proba >= best_threshold else "NORMALE"
        
        # SHAP explanation
        shap_values = explainer.shap_values(data)
        
        # Top 3 features
        feature_importance = np.abs(shap_values[0])
        top_indices = np.argsort(feature_importance)[-3:][::-1]
        top_features = [
            {
                "feature": data.columns[i],
                "value": float(data.iloc[0, i]) if hasattr(data.iloc[0, i], '__len__') else float(data.iloc[0, i]),
                "shap_value": float(shap_values[0, i]),
                "impact": "positif (fraude)" if shap_values[0, i] > 0 else "négatif (normale)"
            }
            for i in top_indices
        ]
        
        return PredictionResponse(
            transaction_id="TX_" + str(hash(str(transaction.dict())))[-8:],
            fraud_score=float(proba),
            prediction=prediction,
            risk_level=get_risk_level(proba),
            top_features=top_features
        )
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('src/api.py', 'w', encoding='utf-8') as f:
    f.write(api_code)

print('✅ src/api.py créé avec succès')

In [ ]:
# === Test de l'API en local ===
print('Pour lancer l\'API depuis Colab :')
print('  1. Exécuter dans un terminal : uvicorn src.api:app --reload')
print('  2. http://127.0.0.1:8000/docs (Swagger UI)')
print('  3. http://127.0.0.1:8000/redoc (Redoc)')

In [ ]:
# === Test via Python ===
print('Test de l\'API avec requests :')
print('''
import requests
import json

url = "http://127.0.0.1:8000/predict"
payload = {
    "TransactionAmt": 50000.0,
    "card1": 12345.0,
    "card2": 100.0,
    "addr1": 200.0,
    "addr2": 50.0,
    "hour": 14,
    "dayofweek": 3
}
response = requests.post(url, json=payload)
print(json.dumps(response.json(), indent=2))
''')
print()
print('Résultat attendu :')
print('{"transaction_id": "TX_abc123", "fraud_score": 0.92,')

In [ ]:
# === Schéma d'architecture (texte) ===
print('''
=== ARCHITECTURE API FRAUDX ===

┌─────────────┐     ┌─────────────────┐     ┌─────────────┐
│  Client      │────>│  API FastAPI    │────>│  Modèle     │
│  (curl/post) │     │  /predict       │     │  XGBoost    │
└─────────────┘     └─────────────────┘     └──────┬──────┘
                                                    │
                          ┌─────────────────────────┘
                          │
                    ┌─────▼──────┐
                    │  SHAP      │
                    │  Explainer │
                    └─────┬──────┘
                          │
                    ┌─────▼──────────────────┐
                    │  Réponse JSON          │
                    │  - fraud_score         │
                    │  - prediction          │
                    │  - risk_level          │
                    │  - top_features (SHAP) │
                    └────────────────────────┘
''')
print('Schéma à reproduire dans draw.io ou LucidChart pour le mémoire (section 3.4)')

---
## Synthèse Phase 7 — Éléments pour le mémoire

### Section 3.4 — Proposition de plateforme (PoC)

**Livrables :**
1. Schéma d'architecture technique (à réaliser dans draw.io)
2. API FastAPI fonctionnelle (`src/api.py`) avec endpoints :
   - `GET /` → statut du service
   - `POST /predict` → score de fraude + SHAP
   - `GET /health` → santé du service
3. Spécifications de la gestion des utilisateurs (rôles) :
   - **Analyste fraude** : visualisation des alertes, validation
   - **Gestionnaire de risques** : configuration des seuils, rapports
   - **Administrateur** : gestion des utilisateurs, audit
4. Mockups du tableau de bord (à réaliser dans Figma/Canva)

### Principes de sécurité (spécifications) :
- Chiffrement TLS/HTTPS
- Authentification JWT (roles)
- Journalisation des accès (audit trail)
- Conformité KYC/AML et BCEAO